In [2]:
import pandas as pd
from pathlib import Path

EXTRACT_DIR = Path(r"D:\07_Data_Science\05_DS_Projects\03_AirCast\data\raw")

# peek at ONE station file
sample = pd.read_csv(EXTRACT_DIR / "AP001.csv")

print("SHAPE:", sample.shape)
print("\nCOLUMNS:", list(sample.columns))
print("\nDTYPES:\n", sample.dtypes)
print("\nHEAD:\n", sample.head())


meta = pd.read_csv(EXTRACT_DIR / "stations_info.csv")
print(meta.shape)
print(list(meta.columns))
print(meta.head(10))

SHAPE: (59150, 23)

COLUMNS: ['From Date', 'To Date', 'PM2.5 (ug/m3)', 'PM10 (ug/m3)', 'NO (ug/m3)', 'NO2 (ug/m3)', 'NOx (ppb)', 'NH3 (ug/m3)', 'SO2 (ug/m3)', 'CO (mg/m3)', 'Ozone (ug/m3)', 'Benzene (ug/m3)', 'Toluene (ug/m3)', 'Temp (degree C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)', 'Xylene (ug/m3)', 'RF (mm)', 'AT (degree C)']

DTYPES:
 From Date           object
To Date             object
PM2.5 (ug/m3)      float64
PM10 (ug/m3)       float64
NO (ug/m3)         float64
NO2 (ug/m3)        float64
NOx (ppb)          float64
NH3 (ug/m3)        float64
SO2 (ug/m3)        float64
CO (mg/m3)         float64
Ozone (ug/m3)      float64
Benzene (ug/m3)    float64
Toluene (ug/m3)    float64
Temp (degree C)    float64
RH (%)             float64
WS (m/s)           float64
WD (deg)           float64
SR (W/mt2)         float64
BP (mmHg)          float64
VWS (m/s)          float64
Xylene (ug/m3)     float64
RF (mm)            float64
AT (degree C)      float64
dt

In [3]:
sample["From Date"] = pd.to_datetime(sample["From Date"], errors="coerce")
print(sample["From Date"].head(5))
print("Gap between rows:", sample["From Date"].iloc[1] - sample["From Date"].iloc[0])

0   2016-07-01 10:00:00
1   2016-07-01 11:00:00
2   2016-07-01 12:00:00
3   2016-07-01 13:00:00
4   2016-07-01 14:00:00
Name: From Date, dtype: datetime64[ns]
Gap between rows: 0 days 01:00:00


In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # if notebook is in notebooks/, use Path.cwd().parent

from src.data.loader import DataLoader

loader = DataLoader(cities=["Guwahati", "Tirupati"])   # small subset for dev
df = loader.load()

print("SHAPE:", df.shape)
print("\nCOLUMNS:", list(df.columns))
print("\nDTYPES:\n", df.dtypes)
print("\nCITIES:", df["city"].unique())
print("\nDup station-hours:", df.duplicated(subset=["station", "datetime"]).sum())
print("\nHEAD:\n", df.head())

SHAPE: (125105, 16)

COLUMNS: ['datetime', 'station', 'city', 'state', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'o3', 'benzene', 'toluene', 'xylene']

DTYPES:
 datetime    datetime64[ns]
station             object
city                object
state               object
pm25               float64
pm10               float64
no                 float64
no2                float64
nox                float64
nh3                float64
so2                float64
co                 float64
o3                 float64
benzene            float64
toluene            float64
xylene             float64
dtype: object

CITIES: ['Guwahati' 'Tirupati']

Dup station-hours: 0

HEAD:
              datetime station      city  state   pm25    pm10     no    no2  \
0 2019-03-01 00:00:00   AS001  Guwahati  Assam  84.50  122.29  17.16  10.45   
1 2019-03-01 01:00:00   AS001  Guwahati  Assam  46.73   58.81  12.54   9.09   
2 2019-03-01 02:00:00   AS001  Guwahati  Assam  40.54   53.46  11.16   8.20   


In [1]:
from src.data.loader import DataLoader
from src.data.aggregator import DailyAggregator

hourly = DataLoader(cities=["Guwahati", "Tirupati"]).load()
daily = DailyAggregator().aggregate(hourly)

print("HOURLY:", hourly.shape)
print("DAILY :", daily.shape)
print("\nCOLUMNS:", daily.columns.tolist())
print("\nDup station-days:", daily.duplicated(subset=["station", "date"]).sum())
print("\nMax hours in a day (must be <= 24):")
print(daily[["pm25_n", "co_n", "o3_n"]].max())
print("\nHEAD:\n", daily.head())

HOURLY: (125105, 16)
DAILY : (5215, 23)

COLUMNS: ['city', 'state', 'station', 'date', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'o3', 'benzene', 'toluene', 'xylene', 'pm25_n', 'pm10_n', 'so2_n', 'no2_n', 'nh3_n', 'co_n', 'o3_n']

Dup station-days: 0

Max hours in a day (must be <= 24):
pm25_n    24
co_n      24
o3_n      24
dtype: int64

HEAD:
        city  state station       date        pm25        pm10         no  \
0  Guwahati  Assam   AS001 2019-03-01   69.676250  108.627083  19.510833   
1  Guwahati  Assam   AS001 2019-03-02   91.920833  157.108750  40.142917   
2  Guwahati  Assam   AS001 2019-03-03  107.808750  200.122083  49.575417   
3  Guwahati  Assam   AS001 2019-03-04   90.682500  178.370417  39.751667   
4  Guwahati  Assam   AS001 2019-03-05   69.123333  115.809583   3.118750   

         no2         nox        nh3  ...   benzene  toluene  xylene  pm25_n  \
0  11.906250   44.794167  22.969583  ...  2.157917      NaN     NaN      24   
1  19.143750   86.40083

In [2]:
from src import config

band = config.AQI_BREAKPOINTS["pm25"][3]   # the 91-120 band
# band = config.AQI_BREAKPOINTS   # the 91-120 band
# print(band)
c_low, c_high, i_low, i_high = band
c = 110
sub = (i_high - i_low) / (c_high - c_low) * (c - c_low) + i_low
print("PM2.5=110 sub-index:", round(sub, 2))   # expect ~265.86

# structural checks
assert set(config.CPCB_AQI_POLLUTANTS) == set(config.AQI_BREAKPOINTS), "pollutant keys must match"
for p, bands in config.AQI_BREAKPOINTS.items():
    assert all(len(b) == 4 for b in bands), f"{p} has a malformed band"
print("All breakpoint tables well-formed ✅")

PM2.5=110 sub-index: 265.86
All breakpoint tables well-formed ✅


In [1]:
from src.data.loader import DataLoader
from src.data.aggregator import DailyAggregator
from src.data.aqi import AQICalculator

hourly  = DataLoader(cities=["Guwahati", "Tirupati"]).load()
daily   = DailyAggregator().aggregate(hourly)
labeled = AQICalculator().compute(daily)

print(labeled[["city", "date", "aqi", "aqi_bucket"]].head())
print("\nAQI range:", round(labeled["aqi"].min(), 1), "-", round(labeled["aqi"].max(), 1))
print("Days with no AQI:", labeled["aqi"].isna().sum())
print("\nCategory spread:\n", labeled["aqi_bucket"].value_counts(dropna=False))

       city       date         aqi aqi_bucket
0  Guwahati 2019-03-01  130.618922   Moderate
1  Guwahati 2019-03-02  204.143534       Poor
2  Guwahati 2019-03-03  258.381595       Poor
3  Guwahati 2019-03-04  199.916121   Moderate
4  Guwahati 2019-03-05  128.731379   Moderate

AQI range: 5.1 - 500.0
Days with no AQI: 124

Category spread:
 aqi_bucket
Satisfactory    2074
Good            1224
Moderate        1027
Very Poor        418
Poor             335
NaN              124
Severe            13
Name: count, dtype: int64


In [1]:
import pandas as pd
from src import config

df = pd.read_parquet(config.DAILY_AQI_PATH)
print(df.shape)
print(df["city"].nunique(), "cities")
print(df["aqi_bucket"].value_counts(dropna=False))

(595882, 25)
241 cities
aqi_bucket
Moderate        164592
Satisfactory    143734
None            113469
Good             58350
Very Poor        53818
Poor             46692
Severe           15227
Name: count, dtype: int64


In [2]:
import pandas as pd
daily = pd.read_parquet(r"D:\07_Data_Science\05_DS_Projects\03_AirCast\data\processed\daily_aqi.parquet")
# daily.head()        # eyeball the top rows
print(daily.sample(10))    # 10 random rows — better for spotting variety

,city,state,station,date,pm25,pm10,no,no2,nox,nh3,...,xylene,pm25_n,pm10_n,so2_n,no2_n,nh3_n,co_n,o3_n,aqi,aqi_bucket
85051,Bilaspur,Chhattisgarh,CG001,2023-03-28,36.046667,87.797083,6.095000,10.962083,17.730833,NaN,...,NaN,24,24,24,24,0,24,0,125.107753,Moderate
315652,Hyderabad,Telangana,TG001,2016-06-20,29.581667,NaN,6.355833,10.865833,17.221667,NaN,...,0.16375,24,0,24,24,0,24,24,109.161234,Moderate
386769,Korba,Chhattisgarh,CG003,2023-02-22,67.286316,162.310000,22.674583,40.899583,39.551667,49.696250,...,NaN,19,19,24,24,24,0,24,141.736174,Moderate
445501,Mumbai,Maharashtra,MH018,2021-10-31,52.105000,178.616250,54.608333,43.097083,97.705833,15.583333,...,NaN,24,24,24,24,24,24,24,152.570529,Moderate
196355,Delhi,Delhi,DL015,2019-03-01,99.010417,277.166667,30.714583,101.991667,63.355833,31.597917,...,NaN,24,24,24,24,24,24,24,228.345905,Poor
237137,Delhi,Delhi,DL035,2020-05-22,127.312500,291.708333,3.139167,40.675417,24.680833,35.289167,...,NaN,24,24,0,24,24,24,24,305.844477,Very Poor
8653,Agra,Uttar Pradesh,UP045,2022-05-06,32.623750,91.203333,16.958333,40.471250,32.353333,38.149583,...,NaN,24,24,24,24,24,24,16,91.203333,Satisfactory
390855,Kurukshetra,Haryana,HR015,2020-07-13,22.023478,59.942174,1.631739,11.575652,7.298261,34.330870,...,NaN,23,23,22,23,23,0,23,59.942174,Satisfactory
536724,Rohtak,Haryana,HR004,2018-09-26,39.572105,NaN,NaN,NaN,NaN,NaN,...,NaN,19,0,19,0,0,19,19,65.483902,Satisfactory
200651,Delhi,Delhi,DL016,2021-08-05,33.344167,94.947083,21.930833,36.965833,58.890833,NaN,...,NaN,24,24,0,24,0,24,24,94.947083,Satisfactory
